# AMP Training Pipeline: Antimicrobial Peptide Activity Prediction

This notebook trains ML models to predict antibacterial activity of antimicrobial peptides (AMPs).

**Workflow:**
1. Data Loading (DBAASP) → Preprocessing → Featurization
2. Model Training (RandomForest / ESM-2)
3. Evaluation & Interpretation (SHAP)
4. Prediction & Ranking

---

## 1. Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from collections import Counter
from typing import List, Dict, Tuple, Optional

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report, 
    matthews_corrcoef, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import ProtParam
from Bio.SeqUtils.ProtParam import ProteinAnalysis

sys.path.append('..')
from modules.amp_module import AMPFeaturizer, AMPRanker

## 2. Data Loading (DBAASP)

### 2.1 How to Download DBAASP Data

**Steps to obtain the dataset:**

1. Go to [dbaasp.org](https://dbaasp.org)
2. Use the **Advanced Search** tool
3. Filter by:
   - Activity: Antibacterial
   - MIC (minimum inhibitory concentration): < 100 µg/mL
   - Peptide type: Linear or Cyclic
   - Length: 10-50 amino acids
4. Click **Search** → **Export** as CSV or FASTA
5. Save the file to `data/amp/dbaasp_export.csv`

**Note:** The free version contains ~23,000+ curated entries. For this notebook, we use a sample dataset structure.

In [ ]:
DATA_DIR = Path('../data/amp')
RESULTS_DIR = Path('../results/amp')
MODELS_DIR = Path('../models')

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_dbaasp_data(filepath: str) -> pd.DataFrame:
    """
    Load DBAASP export file (CSV or FASTA format).
    
    Expected CSV columns:
    - sequence: amino acid sequence
    - mic: minimum inhibitory concentration (µg/mL)
    - target: target organism (bacteria name)
    - activity: activity type (antibacterial, antifungal, etc.)
    """
    filepath = Path(filepath)
    
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}")
        print("Generating synthetic data for demonstration...")
        return generate_synthetic_amp_data(n_samples=2000)
    
    if filepath.suffix == '.csv':
        df = pd.read_csv(filepath)
    elif filepath.suffix == '.fasta':
        records = list(SeqIO.parse(filepath, 'fasta'))
        data = []
        for record in records:
            data.append({
                'sequence': str(record.seq),
                'name': record.name,
                'description': record.description
            })
        df = pd.DataFrame(data)
    else:
        raise ValueError(f"Unsupported file format: {filepath.suffix}")
    
    return df

In [ ]:
def generate_synthetic_amp_data(n_samples: int = 2000) -> pd.DataFrame:
    """
    Generate synthetic AMP data for demonstration.
    In production, replace with actual DBAASP data.
    """
    from modules.amp_module import AMPGenerator
    
    np.random.seed(42)
    
    n_active = n_samples // 2
    n_inactive = n_samples - n_active
    
    active_sequences = AMPGenerator.generate_helical_sequences(n=n_active)
    inactive_sequences = [
        AMPGenerator.generate_random_sequence(10, 50) 
        for _ in range(n_inactive)
    ]
    
    all_sequences = active_sequences + inactive_sequences
    labels = [1] * n_active + [0] * n_inactive
    
    mic_values = []
    for label in labels:
        if label == 1:
            mic_values.append(np.random.uniform(1, 32))
        else:
            mic_values.append(np.random.uniform(64, 256))
    
    target_organisms = ['E. coli', 'S. aureus', 'P. aeruginosa', 
                        'B. subtilis', 'K. pneumoniae'] * (n_samples // 5)
    
    df = pd.DataFrame({
        'sequence': all_sequences,
        'activity': labels,
        'mic': mic_values,
        'target': target_organisms[:n_samples]
    })
    
    return df

In [ ]:
df = load_dbaasp_data(DATA_DIR / 'dbaasp_export.csv')
print(f"Loaded {len(df)} sequences")
df.head()

## 3. Data Preprocessing

In [ ]:
VALID_AMINO_ACIDS = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(seq: str) -> str:
    """Remove non-standard amino acids."""
    return ''.join([aa for aa in seq.upper() if aa in VALID_AMINO_ACIDS])

def create_binary_label(mic_value: float, threshold: float = 32.0) -> int:
    """
    Create binary label from MIC value.
    Active (1): MIC <= threshold µg/mL
    Inactive (0): MIC > threshold µg/mL
    """
    return 1 if mic_value <= threshold else 0

def preprocess_amp_data(df: pd.DataFrame, 
                        mic_threshold: float = 32.0,
                        min_length: int = 10,
                        max_length: int = 50) -> pd.DataFrame:
    """
    Preprocess AMP dataset:
    - Clean sequences
    - Filter by length
    - Create binary labels
    """
    df_clean = df.copy()
    
    if 'sequence' not in df_clean.columns:
        raise ValueError("Dataset must contain 'sequence' column")
    
    df_clean['sequence'] = df_clean['sequence'].astype(str).apply(clean_sequence)
    df_clean = df_clean[df_clean['sequence'].str.len() >= min_length]
    df_clean = df_clean[df_clean['sequence'].str.len() <= max_length]
    
    if 'mic' in df_clean.columns:
        df_clean['activity'] = df_clean['mic'].apply(
            lambda x: create_binary_label(x, mic_threshold)
        )
    elif 'activity' in df_clean.columns:
        if df_clean['activity'].dtype == object:
            df_clean['activity'] = df_clean['activity'].map({'active': 1, 'inactive': 0})
    else:
        print("⚠️  No MIC or activity column found. Using existing 'activity' or generating labels.")
    
    df_clean = df_clean.dropna(subset=['sequence', 'activity'])
    df_clean = df_clean.drop_duplicates(subset=['sequence'])
    
    return df_clean.reset_index(drop=True)

In [ ]:
df_processed = preprocess_amp_data(df, mic_threshold=32.0)
print(f"Processed {len(df_processed)} sequences")
print(f"\nClass distribution:")
print(df_processed['activity'].value_counts())
print(f"\nSequence length distribution:")
print(df_processed['sequence'].str.len().describe())

## 4. Featurization

### 4.1 Physicochemical Descriptors (Baseline)

In [ ]:
def calculate_biopython_descriptors(sequence: str) -> Dict:
    """
    Calculate descriptors using BioPython.
    """
    try:
        analysis = ProteinAnalysis(sequence)
        
        descriptors = {
            'length': len(sequence),
            'molecular_weight': analysis.molecular_weight(),
            'aromaticity': analysis.aromaticity(),
            'isoelectric_point': analysis.isoelectric_point(),
            'gravy': analysis.gravy(),
            'charge_at_pH7': analysis.charge_at_pH(7.0),
            'charge_at_pH5': analysis.charge_at_pH(5.0),
            'helix_frac': analysis.secondary_structure_fraction()[0],
            'turn_frac': analysis.secondary_structure_fraction()[1],
            'sheet_frac': analysis.secondary_structure_fraction()[2],
        }
        
        aa_counts = analysis.count_amino_acids()
        total = sum(aa_counts.values())
        
        for aa in 'ACDEFGHIKLMNPQRSTVWY':
            descriptors[f'aa_{aa}_ratio'] = aa_counts.get(aa, 0) / total if total > 0 else 0
        
        return descriptors
    except Exception as e:
        return {}

def featurize_sequences(sequences: List[str]) -> pd.DataFrame:
    """
    Featurize AMP sequences using both AMPFeaturizer and BioPython.
    """
    all_features = []
    
    for seq in sequences:
        amp_features = AMPFeaturizer.calculate_descriptors(seq)
        bio_features = calculate_biopython_descriptors(seq)
        
        combined = {**amp_features, **bio_features}
        combined['sequence'] = seq
        all_features.append(combined)
    
    return pd.DataFrame(all_features)

In [ ]:
print("Featurizing sequences...")
features_df = featurize_sequences(df_processed['sequence'].tolist())
print(f"Generated {len(features_df.columns)} features")
features_df.head()

### 4.2 ESM-2 Embeddings (Advanced)

In [ ]:
def get_esm2_embeddings(sequences: List[str], 
                        model_name: str = "facebook/esm2_t6_8M_UR50D") -> np.ndarray:
    """
    Generate ESM-2 embeddings for sequences.
    
    Note: Requires transformers and torch installed.
    For CPU, use smaller model: esm2_t6_8M_UR50D
    For GPU, can use: esm2_t12_35M_UR4D or esm2_t30_150M_UR50D
    """
    try:
        from transformers import AutoTokenizer, AutoModel
        import torch
        
        print(f"Loading ESM-2 model: {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        model.eval()
        
        embeddings = []
        
        with torch.no_grad():
            for i, seq in enumerate(sequences):
                if i % 100 == 0:
                    print(f"Processing {i}/{len(sequences)}...")
                
                inputs = tokenizer(seq, return_tensors="pt", 
                                 padding=True, truncation=True, 
                                 max_length=60)
                
                outputs = model(**inputs)
                
                seq_embedding = outputs.last_hidden_state[:, 0, :].numpy().flatten()
                embeddings.append(seq_embedding)
        
        return np.array(embeddings)
    
    except ImportError:
        print("⚠️  transformers/torch not installed. Using physicochemical features only.")
        print("Install with: pip install transformers torch")
        return None

In [ ]:
USE_ESM2 = False  # Set to True if you have GPU or want to try ESM-2

if USE_ESM2:
    print("Generating ESM-2 embeddings (this may take a while on CPU)...")
    esm_embeddings = get_esm2_embeddings(df_processed['sequence'].tolist())
    print(f"ESM-2 embeddings shape: {esm_embeddings.shape}")
else:
    print("Using physicochemical descriptors only (set USE_ESM2=True to enable ESM-2)")
    esm_embeddings = None

## 5. Model Training

In [ ]:
feature_cols = [col for col in features_df.columns if col != 'sequence']
X = features_df[feature_cols].values
y = df_processed['activity'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")
print(f"Class balance - Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE: {len(X_train_balanced)} samples")
print(f"Class balance: {np.bincount(y_train_balanced)}")

### 5.1 Random Forest Classifier (Baseline)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_balanced, y_train_balanced)

y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Random Forest Results:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}")

### 5.2 Gradient Boosting Classifier (Alternative)

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    min_samples_split=5,
    random_state=42
)

gb_model.fit(X_train_balanced, y_train_balanced)

y_pred_gb = gb_model.predict(X_test_scaled)
y_prob_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

print("Gradient Boosting Results:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_gb):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_gb):.4f}")

### 5.3 Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_model, X_train_scaled, y_train, 
    cv=cv, scoring='roc_auc'
)

print(f"5-Fold CV AUC-ROC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Individual folds: {cv_scores}")

## 6. Evaluation & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

from sklearn.metrics import roc_curve, precision_recall_curve

roc_curve_data = roc_curve(y_test, y_prob)
axes[0].plot(roc_curve_data[0], roc_curve_data[1], 'b-', label=f'RF (AUC={roc_auc_score(y_test, y_prob):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

pr_curve_data = precision_recall_curve(y_test, y_prob)
axes[1].plot(pr_curve_data[0], pr_curve_data[1], 'r-', label=f'PR (AP={average_precision_score(y_test, y_prob):.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evaluation_metrics.png', dpi=150)
plt.show()

### 6.1 SHAP Interpretation

In [ ]:
try:
    import shap
    
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_scaled)
    
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, 
                     show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'shap_summary.png', dpi=150)
    plt.show()
    
except ImportError:
    print("⚠️  SHAP not installed. Install with: pip install shap")
    print("Feature importance from Random Forest:")
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(feature_importance.head(15))

## 7. Prediction & Ranking

In [ ]:
def predict_amp_activity(sequences: List[str], model, scaler, feature_cols: List[str]) -> pd.DataFrame:
    """Predict antibacterial activity for new AMP sequences."""
    features = featurize_sequences(sequences)
    X = features[feature_cols].values
    X_scaled = scaler.transform(X)
    
    y_pred = model.predict(X_scaled)
    y_prob = model.predict_proba(X_scaled)[:, 1]
    
    results = features.copy()
    results['predicted_activity'] = y_pred
    results['probability_active'] = y_prob
    
    return results

In [ ]:
new_sequences = [
    "KWLKGLFKIASKKCK",  
    "KFFKRLFQSVPMPVV",
    "AAAAAAAAAAAAAAA",
    "GLFDIVKKVVFVGSL",
]

predictions = predict_amp_activity(new_sequences, rf_model, scaler, feature_cols)
ranked_predictions = AMPRanker.rank_candidates(predictions)

print("Predicted AMP Candidates (Ranked):")
ranked_predictions[['sequence', 'probability_active', 'viability_score', 'hydrophobicity', 'net_charge']]

## 8. Save Model & Results

In [ ]:
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_data = {
    'model': rf_model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'model_type': 'RandomForest',
    'training_date': timestamp,
    'metrics': {
        'auc_roc': float(roc_auc_score(y_test, y_prob)),
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred))
    }
}

model_path = MODELS_DIR / f'amp_model_{timestamp}.joblib'
joblib.dump(model_data, model_path)
print(f"Model saved to: {model_path}")

predictions_path = RESULTS_DIR / f'amp_predictions_{timestamp}.csv'
ranked_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to: {predictions_path}")

## 9. Generative Extension (Optional)

### 9.1 Simple Motif-Based Generation

For more advanced generative models, consider:
- **AMP-GAN**: Train a GAN on AMP sequences
- **REINVENT**: RL-based generation with predictor rewards
- **ESM-2 sampling**: Fine-tune ESM-2 for conditional generation

In [ ]:
def generate_motif_variants(motif: str, n_variants: int = 100) -> List[str]:
    """
    Generate variants from a motif pattern.
    X = any amino acid
    """
    from modules.amp_module import AMINO_ACIDS, AMPGenerator
    
    variants = []
    motif_len = len(motif)
    
    for _ in range(n_variants):
        variant = list(motif)
        for i, aa in enumerate(variant):
            if aa == 'X':
                variant[i] = random.choice(AMINO_ACIDS)
        variants.append(''.join(variant))
    
    return variants

motif = "KXKXXXXKXK"
generated = generate_motif_variants(motif, 50)
print(f"Generated {len(generated)} variants from motif '{motif}'")
print(f"Example: {generated[:5]}")

### 9.2 Predictor-Guided Generation (Mock RL)
```python
# Pseudocode for RL-guided generation:

for iteration in range(n_iterations):
    batch = generate_random_sequences(batch_size)
    scores = model.predict(batch)
    
    # Reward high activity + drug-likeness
    rewards = scores * synthesizability_filter(batch)
    
    # Update generator (mock - would use actual RL in production)
    top_sequences = batch[np.argsort(rewards)[-top_k:]]
```

## Summary

This notebook demonstrates:
1. **Data loading** from DBAASP (or synthetic fallback)
2. **Preprocessing** (cleaning, labeling, balancing)
3. **Featurization** (physicochemical + optional ESM-2)
4. **Model training** (Random Forest, Gradient Boosting)
5. **Evaluation** (AUC-ROC, PR curves, confusion matrix)
6. **Interpretation** (SHAP/feature importance)
7. **Prediction & ranking** of new candidates

**Next steps:**
- Load real DBAASP data
- Train with ESM-2 embeddings (GPU recommended)
- Add multi-task prediction (toxicity, hemolysis)
- Integrate with core pipeline for composite ranking